In [1]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.types import *

In [2]:
# init spark

spark = (
    SparkSession.builder
      .config("spark.driver.memory", "48g")                 # ajuste p/ 16g–24g
      .config("spark.sql.execution.arrow.pyspark.enabled", "true")
      .config("spark.sql.execution.arrow.maxRecordsPerBatch", "20000")
      .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)  # 64MB/partição
      .config("spark.driver.maxResultSize", "0")            # sem limite de resultado (cuidado)
      .getOrCreate()
)

spark.version

'4.0.1'

### read exam features table

In [3]:
# read table
data = spark.read.option("header", True).csv(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/dados/exames.csv"
    )

In [4]:
# drop nas in "Valor Resultado" column
data = data.filter(F.col("Valor Resultado").isNotNull())

In [5]:
data.printSchema()

root
 |-- Data Solicitação: string (nullable = true)
 |-- Unidade Solicitante: string (nullable = true)
 |-- Solicitação: string (nullable = true)
 |-- Item Solicitado: string (nullable = true)
 |-- Exame Solicitado: string (nullable = true)
 |-- Material Análise: string (nullable = true)
 |-- dthr_entrada: string (nullable = true)
 |-- Data Liberação: string (nullable = true)
 |-- Prontuário: string (nullable = true)
 |-- Origem: string (nullable = true)
 |-- Nome do Parâmetro: string (nullable = true)
 |-- Valor Resultado: string (nullable = true)
 |-- Sequencial Resultado Item: string (nullable = true)



In [6]:
data.select("Prontuário").distinct().count()

57012

### Criar variavel com informacoes de hemograma padronizadas

In [7]:
params = {
    "PLAQUETAS": "PLAQUETAS",
    "NEUTRÓFILOS SEGMENTADOS": "NEUTROFILOS_SEGMENTADOS",
    "EOSINÓFILOS": "EOSINOFILOS",
    "BASÓFILOS": "BASOFILOS",
    "LINFÓCITOS": "LINFOCITOS",
    "LINFÓCITO ATÍPICO": "LINFOCITO_ATIPICO"
}

data = data.withColumn("param_final", F.lit(None).cast("string"))
for key, val in params.items():
    data = data.withColumn(
        "param_final",
        F.when(F.upper(F.col("Nome do Parâmetro")).contains(key.upper()), val)
        .otherwise(F.col("param_final"))
    )

In [8]:
# drop nas in "Valor Resultado" column
data = data.filter(F.col("param_final").isNotNull())

In [9]:
# tratamentos dos valores das categorias que nao sao plaqueta

def dividePor100(data, param_column, param_100):
    """ divide por 100 a coluna 'Valor Resultado', 
        apenas nos casos em que a coluna param_100 == param_100
        pressupoe que o vies de ,00 virar algarismo numerico acontece em categorias especificas de 
        parametro de acordo com a estrutura do dataset
    """

    data = data.withColumn(
        "Valor Resultado",
    F.when(
        F.col("Nome do Parâmetro") == param_100,
        F.col("Valor Resultado") / 100
    ).otherwise(F.col("Valor Resultado"))
    
    )

    # verificar
    #paramdata = data[data["param_final"] == param_column]
    #print(paramdata.select(["Nome do Parâmetro", "Valor Resultado"]).groupBy(["Nome do Parâmetro"]).mean().show(20, False))

    return data

In [10]:
data = data.withColumn(
    "Valor Resultado",
    F.when(
        (F.col("param_final") == 'LINFOCITOS') &
        (F.col("Valor Resultado") > 20000),
        F.col("Valor Resultado") / 100
    ).otherwise(F.col("Valor Resultado"))
)

In [11]:
data = dividePor100(data, 
             param_column="NEUTROFILOS_SEGMENTADOS", 
             param_100="[NEUTRÓFILOS SEGMENTADOS %1] * [LEUCÓCITOS1] / 100")

data = dividePor100(data, 
             param_column="BASOFILOS", 
             param_100="[BASÓFILOS %1] * [LEUCÓCITOS1] / 100")

data = dividePor100(data, 
             param_column="EOSINOFILOS", 
             param_100="[EOSINÓFILOS %1] * [LEUCÓCITOS1] / 100")

data = dividePor100(data, 
             param_column="BASOFILOS", 
             param_100="[BASÓFILOS %1] * [LEUCÓCITOS1] / 100")

In [12]:
def generate_exam_features(data):

    # groupby average value by prontuario + param_final
    w_prompt_param = W.partitionBy("Prontuário", "param_final")

    agg_mean =( 
               (data.withColumn("valor_medio", F.mean("Valor Resultado").over(w_prompt_param))
                .select(
                    F.col("Prontuário").alias("prontuario"),
                    "param_final",
                    "valor_medio"
                    )
        ).groupBy("prontuario")
        .pivot("param_final")
        .agg(F.first("valor_medio"))
    )

    
    # renomeamos as colunas para adicionar os sufixos
    for col in agg_mean.columns[1:]:
        agg_mean = agg_mean.withColumnRenamed(col, f"{col.lower()}_valor_medio")

    # groupby max date value by prontuario + param_final
    data = data.withColumn("data_lib_ts", F.to_timestamp(F.col("Data Liberação")))  # garante timestamp
    w_orderdate = W.partitionBy("Prontuário", "param_final").orderBy(F.col("data_lib_ts").desc())

    agg_max_date = (
                (data.withColumn("rn", F.row_number().over(w_orderdate))
                .filter(F.col("rn") == 1)
                .select(
                    F.col("Prontuário").alias("prontuario"),
                    "param_final",
                    F.col("Valor Resultado").alias("valor_max_date")
                )
        ).groupBy("prontuario")
        .pivot("param_final")
        .agg(F.first("valor_max_date"))
    )

    for col in agg_max_date.columns[1:]:
        agg_max_date = agg_max_date.withColumnRenamed(col, f"{col.lower()}_valor_max_date")


    # join the two datasets
    features = agg_mean.join(agg_max_date, on="prontuario", how="inner")

    return features

### Aplicar criação de variáveis por janelas temporais

In [13]:

# construir meses de referencia a partir das datas minima e maxima
bounds = data.agg(
    F.date_trunc("month", F.min("Data Liberação")).alias("min_m"),
    F.date_trunc("month", F.max("Data Liberação")).alias("max_m"),
)

ref_dates_df = bounds.select(
    F.expr("sequence(min_m, max_m, interval 1 month) as ref_dates")
).select(F.explode("ref_dates").alias("ref_date"))

# lista de ref dates distintos
ref_dates = [r.ref_date for r in ref_dates_df.collect()]

In [14]:
# define dataframe para incorporar dados ao cursor
features = spark.createDataFrame([], schema=StructType())

# para cada data de referencia, filtrar os dados anteriores a ela, para calculo das variaveis
for ref_ts in ref_dates:

    data_filtered = data.filter(
        (F.col("Data Liberação") < F.lit(ref_ts)) &
        (F.col("Data Liberação") >= F.add_months(F.lit(ref_ts), -12))
    )
    
    features_ref_date = generate_exam_features(data_filtered)

    features_ref_date = features_ref_date.withColumn("date_ref", F.lit(ref_ts).cast("timestamp"))

    features = features.unionByName(features_ref_date, allowMissingColumns=True)

    print(ref_ts)

2022-07-01 00:00:00
2022-08-01 00:00:00
2022-09-01 00:00:00
2022-10-01 00:00:00
2022-11-01 00:00:00
2022-12-01 00:00:00
2023-01-01 00:00:00
2023-02-01 00:00:00
2023-03-01 00:00:00
2023-04-01 00:00:00
2023-05-01 00:00:00
2023-06-01 00:00:00
2023-07-01 00:00:00
2023-08-01 00:00:00
2023-09-01 00:00:00
2023-10-01 00:00:00
2023-11-01 00:00:00
2023-12-01 00:00:00
2024-01-01 00:00:00
2024-02-01 00:00:00
2024-03-01 00:00:00
2024-04-01 00:00:00
2024-05-01 00:00:00
2024-06-01 00:00:00
2024-07-01 00:00:00
2024-08-01 00:00:00
2024-09-01 00:00:00
2024-10-01 00:00:00
2024-11-01 00:00:00
2024-12-01 00:00:00
2025-01-01 00:00:00
2025-02-01 00:00:00
2025-03-01 00:00:00
2025-04-01 00:00:00
2025-05-01 00:00:00
2025-06-01 00:00:00
2025-07-01 00:00:00
2025-08-01 00:00:00
2025-09-01 00:00:00
2025-10-01 00:00:00


In [16]:
# filter pacientes in target data
pacientes_target = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/sample_target_internacao.parquet"
).select("prontuario").distinct()

pacientes_target.count()

17381

In [17]:
features = features.join(pacientes_target, on="prontuario", how="inner")
features.count()

225886

In [18]:
exames_out = features.toPandas()

In [20]:
exames_out.to_parquet("C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/sample_features_exams.parquet")